# Activity 8 (quarterly version): LSTM on the quarterly data

**Tuesday · Block 3** · Kernel: `nwcst` · About 20 minutes

The hands-on part of the LSTM lecture, run on the same quarterly table as Blocks 1 and 2: one
row per quarter, each predictor averaged over the quarter's three months. For every quarter the
LSTM looks at a window of the last `n_timesteps` quarters and learns which patterns in that
window predict GDP growth.

You will:

1. Put the quarterly data in the shape `nowcast_lstm` expects
2. Fit an LSTM on the train sample and score it on the same 12 test quarters as Blocks 1 and 2
3. Move `train_episodes` and watch under- and overfitting
4. See what bagging (`n_models`) does

Cells marked **YOUR TURN** are yours.

## Preparation

The data are the same files used in Blocks 1 and 2:

| File | What it holds |
|---|---|
| `blocks_quarterly.csv` | Real GDP growth (`RGDP0000`), a `sample` column (`train` / `test`) and 15 predictors |
| `blocks_meta.csv` | Short labels for charts |

Two settings in the cell below matter:

- **CPU only.** `nowcast_lstm` uses a GPU when it finds one; we force the CPU so everyone gets
  the same numbers.
- **Seeds.** Each of the `n_models` networks starts from random weights. Passing a fixed list
  of seeds makes the run reproducible.

Run as is.

In [ ]:
import os
import time
import copy
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from nowcast_lstm.LSTM import LSTM

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(42)
torch.manual_seed(42)

PATH_DATA = os.path.join("..", "data")
PATH_OUTPUT = os.path.join("..", "output")
os.makedirs(PATH_OUTPUT, exist_ok=True)
RESULTS_FILE = os.path.join(PATH_OUTPUT, "workshop_results.csv")
LANGUAGE = "Python"

target_variable = "RGDP0000"

quarterly = pd.read_csv(os.path.join(PATH_DATA, "blocks_quarterly.csv"), parse_dates=["date"])
meta = pd.read_csv(os.path.join(PATH_DATA, "blocks_meta.csv"))

predictors = [c for c in quarterly.columns if c not in ("date", target_variable, "sample")]
LABELS = meta.set_index("ticker")["label"].to_dict()
labels = [LABELS.get(c, c) for c in predictors]

# Same split as Blocks 1-2
train_dates = quarterly.loc[quarterly["sample"] == "train", "date"]
test_dates = quarterly.loc[quarterly["sample"] == "test", "date"]
train_end = train_dates.max()
test_end = test_dates.max()

y_obs = quarterly.set_index("date").loc[test_dates, target_variable]
y_train = quarterly.set_index("date").loc[train_dates, target_variable]

print(f"Quarterly data: {quarterly.date.min():%Y-%m} to {quarterly.date.max():%Y-%m} ({len(quarterly)} quarters)")
print(f"Train sample  : up to {train_end:%Y-%m} ({len(train_dates)} quarters)")
print(f"Test quarters : {test_dates.min():%Y-%m} to {test_end:%Y-%m} ({len(test_dates)})")

The helpers are the same as in Blocks 1 and 2 (`rmse`, `comparison_table`, `save_result`,
`results_table`, `plot_observed_predicted`). Run and move on.

In [ ]:
RESULT_COLS = ["block", "activity", "language", "model", "rmse_test", "notes"]


def rmse(observed, predicted):
    """Root mean squared error."""
    return float(np.sqrt(np.mean((np.asarray(observed) - np.asarray(predicted)) ** 2)))


def comparison_table(predicted, name="Predicted"):
    """Observed vs predicted on the test sample, with the error in percentage points."""
    t = pd.DataFrame({"Observed": y_obs.values, name: np.asarray(predicted)}, index=y_obs.index)
    t["Error (pp)"] = t[name] - t["Observed"]
    return t.round(3)


def save_result(block, activity, model, rmse_value, notes=""):
    """Add or replace one row of the workshop results table (key: activity, language, model).
    Rows written by the R activities are kept untouched."""
    row = pd.DataFrame([{
        "block": int(block), "activity": activity, "language": LANGUAGE,
        "model": model, "rmse_test": round(float(rmse_value), 4), "notes": notes,
    }])
    if os.path.exists(RESULTS_FILE) and os.path.getsize(RESULTS_FILE) > 0:
        prev = pd.read_csv(RESULTS_FILE)
        same = ((prev["activity"] == activity) & (prev["language"] == LANGUAGE)
                & (prev["model"] == model))
        row = pd.concat([prev[~same], row], ignore_index=True)
    os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
    row[RESULT_COLS].to_csv(RESULTS_FILE, index=False)
    print(f"Saved: {model} ({activity}, {LANGUAGE})  RMSE = {float(rmse_value):.4f}")


def results_table():
    """Everything saved so far, by every block and language, best to worst."""
    if not os.path.exists(RESULTS_FILE) or os.path.getsize(RESULTS_FILE) == 0:
        print("No results saved yet.")
        return pd.DataFrame(columns=RESULT_COLS)
    t = pd.read_csv(RESULTS_FILE).sort_values("rmse_test").reset_index(drop=True)
    print(t.to_string())
    return t


def plot_observed_predicted(predicted, name, color="#8E44AD", ax=None):
    """Observed vs predicted GDP growth on the test sample."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    ax.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2, label="Observed")
    ax.plot(y_obs.index, np.asarray(predicted), "s--", color=color, lw=2, label=name)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("Real GDP growth, q/q (%)")
    ax.set_title(f"Observed vs predicted: {name}")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    return ax


results_table();

## The data, the LSTM way

Unlike the models in Blocks 1 and 2, which receive a predictor matrix and a target vector
separately, `nowcast_lstm` receives one table: a `date` column, the predictors and the target
last. Each training example is a window of `n_timesteps` consecutive quarters of all 15
predictors, and its label is the GDP growth of the window's last quarter.

To predict a test quarter the network also needs the quarters before it, so for scoring we pass
the whole history up to the last test quarter, not only the test rows.

**Standardizing.** As in Block 1, the predictors are put on a common scale using the train
sample's mean and standard deviation. Neural networks train poorly when one input moves by 30
and another by 0.3. GDP growth, the target, stays in percent.

In [ ]:
train_mask = quarterly["date"] <= train_end
mu = quarterly.loc[train_mask, predictors].mean()
sd = quarterly.loc[train_mask, predictors].std()

data_std = quarterly[["date"] + predictors + [target_variable]].copy()
data_std[predictors] = (data_std[predictors] - mu) / sd

# Train sample: every quarter up to the last train quarter
train_lstm = data_std[data_std["date"] <= train_end].reset_index(drop=True)
# History up to the last test quarter, for scoring
hist_lstm = data_std[data_std["date"] <= test_end].reset_index(drop=True)

print(f"Train sample: {len(train_lstm)} quarters   Full history: {len(hist_lstm)} quarters")
data_std.tail(5).round(2)

## The knobs

| Parameter | Effect |
|---|---|
| `n_timesteps` | Length of the history window. Longer captures more dynamics but uses up observations. |
| `n_models` | How many networks are trained and averaged. Each starts from different random weights. |
| `train_episodes` | Training epochs per network. |
| `n_hidden` | Hidden units in the cell. More capacity and more risk of overfitting. |
| `n_layers` | Stacked layers. With short series it is best left at 1. |
| `lr` | Optimizer step size. |
| `dropout` | Share of units randomly switched off during training. |

The values used here:

| Parameter | Here | Note |
|---|---|---|
| `n_timesteps` | 4 | The pipeline runs on monthly data with a 12-month window; with quarterly data the same year of history is 4 quarters. Each extra quarter costs one training window. |
| `n_models` | 10 | 50 in the pipeline; 10 so the cells run in seconds. |
| `train_episodes` | 100 (you will move it) | Too few = underfit; too many = memorize. |
| `batch_size` | 50 | Training windows per gradient step. With fewer than 50 windows, every epoch is one step on all of them. |
| `decay` | 0.98 | Per-epoch learning-rate decay. |
| `n_hidden` | 10 | |
| `n_layers` | 1 | |
| `lr` | 0.01 | Initial learning rate of the Adam optimizer. |
| `dropout` | 0.0 | We rely on bagging instead. |
| `seeds` | 0, 1, ..., 9 | One per network, for reproducibility. |

In [ ]:
BASE_PARAMS = dict(
    n_timesteps=4,
    fill_na_func=np.nanmean,
    fill_ragged_edges_func=np.nanmean,
    n_models=10,
    train_episodes=100,
    batch_size=50,
    decay=0.98,
    n_hidden=10,
    n_layers=1,
    dropout=0.0,
    criterion=torch.nn.MSELoss(),
    optimizer=torch.optim.Adam,
    optimizer_parameters={"lr": 1e-2, "weight_decay": 0.0},
)


def fit_lstm(**changes):
    """Fit an LSTM on the train sample with BASE_PARAMS, overridden by `changes`."""
    params = {**BASE_PARAMS, **changes}
    params["seeds"] = list(range(params["n_models"]))       # fixed seeds: reproducible
    model = LSTM(data=train_lstm, target_variable=target_variable, **params)
    model.device = "cpu"                                     # same numbers on every laptop
    model.train(quiet=True)
    return model


def score_lstm(model):
    """Train and test RMSE of a fitted LSTM, plus its test predictions."""
    p = model.predict(hist_lstm).set_index("date")["predictions"]
    pred_test = p.reindex(test_dates).values
    tr = p.reindex(train_dates).dropna()
    return rmse(y_train.loc[tr.index], tr), rmse(y_obs, pred_test), pred_test

## A first LSTM

In [ ]:
t0 = time.time()
lstm_base = fit_lstm()
rmse_base_train, rmse_base, pred_base = score_lstm(lstm_base)

print(f"Fitted {lstm_base.n_models} networks in {time.time() - t0:.1f} s "
      f"on {len(lstm_base.y)} training windows")
print(f"RMSE train : {rmse_base_train:.4f}")
print(f"RMSE test  : {rmse_base:.4f}")
comparison_table(pred_base, "LSTM")

Look at the train RMSE. With 10 hidden units the network has several hundred weights and fewer
than forty training windows, so 100 epochs are enough to learn the train quarters almost by
heart, 2020 included. The test RMSE is what it can actually do.

### Exploring `train_episodes`

The same model with 5 to 200 epochs. As with the depth of a tree, the train error keeps
falling, and the test error tells you where the network stops learning and starts memorizing.

In [ ]:
episodes = [5, 10, 25, 50, 100, 200]
rows = []
t0 = time.time()
for ep in episodes:
    m = fit_lstm(train_episodes=ep)
    r_tr, r_te, _ = score_lstm(m)
    rows.append({"train_episodes": ep, "RMSE train": r_tr, "RMSE test": r_te})
explore_ep = pd.DataFrame(rows)
print(f"{len(episodes)} configurations in {time.time() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(explore_ep["train_episodes"], explore_ep["RMSE train"], "o-", color="#2980B9", lw=2, label="Train")
ax.plot(explore_ep["train_episodes"], explore_ep["RMSE test"], "s-", color="#C0392B", lw=2, label="Test")
ax.set_xscale("log")
ax.set_xlabel("train_episodes (log scale)")
ax.set_ylabel("RMSE")
ax.set_title("LSTM: underfit with too few epochs, memorizes with many")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
plt.tight_layout()

explore_ep.round(4)

Read the left end of the chart carefully. After only a handful of epochs the network has
barely moved from its random start: it predicts something close to the average for every
quarter. That is underfitting. At the right end the train error is close to zero: the network
has memorized the train quarters.

### YOUR TURN: fix the final model

From the table above, pick `train_episodes` and set it here. If there is time, also try
`n_timesteps = 8` (two years instead of one) and compare: each extra quarter in the window is
one less training window. This LSTM is saved as the block's official result. (Same caution as
in Blocks 1 and 2: choosing on the test RMSE is fine for learning; a reported model would be
tuned on the train sample only.)

In [ ]:
# ============ YOUR TURN ============
# Copy here the values you chose.
n_timesteps = 4
train_episodes = 100
# ===================================

t0 = time.time()
lstm = fit_lstm(n_timesteps=n_timesteps, train_episodes=train_episodes)
rmse_lstm_train, rmse_lstm, pred_lstm = score_lstm(lstm)

print(f"Fitted in {time.time() - t0:.1f} s")
print(f"RMSE train : {rmse_lstm_train:.4f}")
print(f"RMSE test  : {rmse_lstm:.4f}   (first LSTM: {rmse_base:.4f})")

plot_observed_predicted(pred_lstm, "LSTM (quarterly)")
save_result(3, "act8", "LSTM (quarterly)", rmse_lstm,
            f"n_timesteps={n_timesteps} quarters, train_episodes={train_episodes}, n_models={lstm.n_models}")

### Chart: Bagging, ten networks, one forecast

`n_models` plays the role of the trees in a random forest. Each network starts from different
random weights and ends somewhere different; the LSTM's forecast is their average. The grey
lines are the ten networks one by one on the test quarters, the purple line is the average.
The right panel shows each network's RMSE against the average's.

In [ ]:
per_net = []
for net in lstm.mv_lstm:
    one = copy.copy(lstm)                 # same data and settings ...
    one.mv_lstm, one.n_models = [net], 1  # ... but a single network
    p = one.predict(hist_lstm).set_index("date")["predictions"]
    per_net.append(p.reindex(test_dates).values)
per_net = np.array(per_net)
rmse_nets = np.array([rmse(y_obs, p) for p in per_net])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
for p in per_net:
    ax1.plot(y_obs.index, p, color="grey", lw=0.8, alpha=0.6)
ax1.plot(y_obs.index, y_obs.values, "o-", color="#2C3E50", lw=2.5, label="Observed")
ax1.plot(y_obs.index, pred_lstm, "s-", color="#8E44AD", lw=2.5, label="Average of the networks")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_ylabel("Real GDP growth, q/q (%)")
ax1.set_title("Each grey line is one network (seed)")
ax1.legend(frameon=False)
ax1.grid(alpha=0.25)

ax2.bar(range(len(rmse_nets)), np.sort(rmse_nets), color="lightgrey", edgecolor="grey")
ax2.axhline(rmse_lstm, color="#8E44AD", lw=2.5, label=f"Average ({rmse_lstm:.2f})")
ax2.set_xlabel("Networks, sorted by RMSE")
ax2.set_ylabel("Test RMSE")
ax2.set_title("Error of each network")
ax2.legend(frameon=False)
plt.tight_layout()

print(f"The average beats {(rmse_nets > rmse_lstm).mean():.0%} of its own networks")

## Wrap-up: the full table

Every model of the three blocks, scored on the same 12 test quarters.

Questions for the group:

- How far apart are the models? On 12 quarters, a few tenths of a point is within what a
  different test window could reverse.
- Here the LSTM saw the same quarterly averages as every other model. In the nowcast tool it
  reads the monthly series directly and handles the months that are still missing at the end
  of the sample, which is its real advantage.

In [ ]:
res = results_table()
if len(res):
    names = res["model"] + " (" + res["language"] + ")"
    block_colors = {0: "#95A5A6", 1: "#2980B9", 2: "#27AE60", 3: "#8E44AD"}
    fig, ax = plt.subplots(figsize=(10, 0.4 * len(res) + 1.5))
    ax.barh(names, res["rmse_test"], color=[block_colors.get(b, "#E67E22") for b in res["block"]])
    ax.invert_yaxis()
    ax.set_xlabel("Test RMSE (pp of q/q GDP growth)")
    ax.set_title("Workshop results on the 12 test quarters")
    ax.grid(alpha=0.25, axis="x")
    plt.tight_layout()